# Polynomial Derivative

> **About this project:** this notebook was built together with **[Claude Code](https://claude.com/claude-code)**, Anthropic's AI coding agent.
> Its purpose is to demonstrate that Claude can be used effectively for this kind of task: I described the goal in plain language,
> and then guided, reviewed and validated the result — design, implementation, tests and publishing to GitHub.
> It shows that I already know how to work with this tool to deliver software faster without giving up on quality.

This notebook takes a simple polynomial written as text and computes its derivative using the **power rule**:

$$\frac{d}{dx}\left(a\,x^n\right) = n\,a\,x^{n-1}$$

Since the derivative of a sum is the sum of the derivatives, it is enough to apply the rule to each term.

**Accepted input format**
- Terms of the form `a·x^n`: a coefficient (integer or decimal), the variable `x` and a non-negative integer exponent.
- Valid examples: `3x^4 - 2x^2 + 5x - 7`, `x^3 + x`, `-x^5 + 2.5x`, `3*x**2 + 1`.
- Spaces, `*` for multiplication and `**` for powers are allowed.
- If an exponent appears more than once (`4x^2 + 3x^2`), the terms are added together.

## 1. Parse the polynomial

The polynomial is represented as a dictionary `{exponent: coefficient}`. For example, `3x^2 - 5x + 1` is stored as `{2: 3, 1: -5, 0: 1}`.

Coefficients are stored as `Fraction` to avoid rounding errors (with `float`, `0.1 * 3` gives `0.30000000000000004`).

In [ ]:
# Project built with Claude Code (Anthropic's AI coding agent) to demonstrate
# that Claude can be used for this purpose: the author directed, reviewed and
# validated the work, showing hands-on experience with the tool.

import re
from fractions import Fraction

# optional sign, optional coefficient, and optionally x with an optional exponent
TERM_PATTERN = re.compile(r"^([+-])?(\d+(?:\.\d+)?)?(x(?:\^(\d+))?)?$")


def parse_polynomial(text: str) -> dict[int, Fraction]:
    """Turn a string like '3x^2 - 5x + 1' into {2: 3, 1: -5, 0: 1}."""
    clean = text.replace(" ", "").replace("**", "^").replace("*", "").lower()
    if not clean:
        raise ValueError("The polynomial is empty.")

    terms = re.findall(r"[+-]?[^+-]+", clean)
    if "".join(terms) != clean:
        raise ValueError(f"Check the signs in '{text}' (there is a loose or repeated '+' or '-').")

    coefficients: dict[int, Fraction] = {}
    for term in terms:
        match = TERM_PATTERN.match(term)
        if not match or not (match.group(2) or match.group(3)):
            raise ValueError(f"Invalid term: '{term}'. Use the form a·x^n, for example 3x^2.")
        sign, number, variable, exponent = match.groups()

        coefficient = Fraction(number) if number else Fraction(1)
        if sign == "-":
            coefficient = -coefficient

        if variable is None:
            power = 0
        elif exponent is None:
            power = 1
        else:
            power = int(exponent)

        coefficients[power] = coefficients.get(power, 0) + coefficient

    # drop terms that cancelled out and keep integer coefficients as int
    return {n: int(a) if a.denominator == 1 else a for n, a in coefficients.items() if a != 0}


parse_polynomial("3x^4 - 2x^2 + 5x - 7")

## 2. Differentiate

Apply the power rule to each term: the coefficient is multiplied by the exponent and the exponent drops by 1. Constant terms (`n = 0`) disappear.

In [ ]:
def differentiate(coefficients: dict[int, Fraction]) -> dict[int, Fraction]:
    """Power rule: a·x^n  ->  (n·a)·x^(n-1)."""
    return {n - 1: n * a for n, a in coefficients.items() if n > 0}


differentiate({4: 3, 2: -2, 1: 5, 0: -7})

## 3. Show the result as text

Convert the dictionary back into readable text, sorted from highest to lowest exponent.

In [ ]:
def format_number(value: Fraction) -> str:
    value = Fraction(value)
    return str(value.numerator) if value.denominator == 1 else str(float(value))


def polynomial_to_text(coefficients: dict[int, Fraction]) -> str:
    """Turn {2: 3, 1: -5, 0: 1} into '3x^2 - 5x + 1'."""
    text = ""
    for n in sorted(coefficients, reverse=True):
        a = coefficients[n]
        if a == 0:
            continue

        magnitude = format_number(abs(a))
        if n == 0:
            body = magnitude
        else:
            coefficient = "" if magnitude == "1" else magnitude
            variable = "x" if n == 1 else f"x^{n}"
            body = coefficient + variable

        if not text:
            text = f"-{body}" if a < 0 else body
        else:
            text += f" - {body}" if a < 0 else f" + {body}"

    return text or "0"


def differentiate_polynomial(text: str) -> str:
    """Take the polynomial as text and return its derivative as text."""
    return polynomial_to_text(differentiate(parse_polynomial(text)))


differentiate_polynomial("3x^4 - 2x^2 + 5x - 7")

## 4. Examples

In [ ]:
examples = [
    "3x^4 - 2x^2 + 5x - 7",
    "x^3 + x",
    "-x^5 + 4x^2 - x",
    "2.5x^3 - 0.5x",
    "3*x**2 + 1",
    "4x^2 + 3x^2 - x",
    "7",
]

for polynomial in examples:
    original = polynomial_to_text(parse_polynomial(polynomial))
    print(f"f(x)  = {original}")
    print(f"f'(x) = {differentiate_polynomial(polynomial)}\n")

## 5. Quick tests

If this cell runs without errors, everything works as expected.

In [ ]:
cases = {
    "3x^4 - 2x^2 + 5x - 7": "12x^3 - 4x + 5",
    "x^3 + x": "3x^2 + 1",
    "-x^2": "-2x",
    "x": "1",
    "7": "0",
    "2.5x^2": "5x",
    "0.1x^3": "0.3x^2",
    "3*x**2 + x": "6x + 1",
    "4x^2 + 3x^2": "14x",
    "x^2 - x^2 + 3": "0",
}
for given, expected in cases.items():
    result = differentiate_polynomial(given)
    assert result == expected, f"{given!r}: expected {expected!r} but got {result!r}"

invalid_inputs = ["", "3x^", "3x2", "3x + + 2", "3y^2", "x^-2"]
for invalid in invalid_inputs:
    try:
        parse_polynomial(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError(f"{invalid!r} should be rejected")

print(f"All tests passed ({len(cases)} derivatives and {len(invalid_inputs)} invalid inputs).")

## 6. Differentiate your own polynomial

Run the cell and type the polynomial when prompted.

In [ ]:
polynomial = input("Enter a polynomial (e.g. 3x^4 - 2x^2 + 5x - 7): ")

try:
    print(f"f(x)  = {polynomial_to_text(parse_polynomial(polynomial))}")
    print(f"f'(x) = {differentiate_polynomial(polynomial)}")
except ValueError as error:
    print(f"Error: {error}")